In [1]:
# 1. Install the compatible version of protobuf
!pip install protobuf==3.20.3

# 2. Install compatible scikit-learn for UMAP
!pip install scikit-learn==1.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tens

In [2]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.0/153.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 72.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 86.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 11.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# --- 0. ENVIRONMENT FIXES (Run this first) ---
import os
import sys

# Fix for the 'MessageFactory' / Protobuf error
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# Fix for the 'ensure_all_finite' UMAP error
try:
    import sklearn.utils.validation
    _original_check_array = sklearn.utils.validation.check_array

    def _patched_check_array(array, *args, **kwargs):
        if 'ensure_all_finite' in kwargs:
            kwargs['force_all_finite'] = kwargs.pop('ensure_all_finite')
        return _original_check_array(array, *args, **kwargs)

    sklearn.utils.validation.check_array = _patched_check_array
    if hasattr(sklearn.utils, 'check_array'):
        sklearn.utils.check_array = _patched_check_array
except ImportError:
    pass

# ------------------------------------------------

import pandas as pd
import numpy as np
import logging
from sklearn.model_selection import train_test_split
from bertopic import BERTopic
from umap import UMAP
import matplotlib.pyplot as plt
import plotly.io as pio

# Set plotly to render in notebook
pio.renderers.default = 'iframe' # 'iframe' is often more robust in Kaggle/Colab than 'notebook'

# Set up basic logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# --- 1. Configuration ---
class Config:
    # Input Text Data Paths
    POS_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair.txt" 
    NEG_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt"
    
    # Embedding & ID Paths
    LBERT_EMB_PATH = "/kaggle/input/lrec-tcs-attack-vs-support-finetuned-embeddings/lbert_finetuned_embeddings.npy"
    LBERT_ID_PATH = "/kaggle/input/lrec-tcs-attack-vs-support-finetuned-embeddings/lbert_finetuned_ids.csv"
    
    INLBERT_EMB_PATH = "/kaggle/input/lrec-tcs-attack-vs-support-finetuned-embeddings/bert_finetuned_embeddings.npy"
    INLBERT_ID_PATH = "/kaggle/input/lrec-tcs-attack-vs-support-finetuned-embeddings/Inlbert_finetuned_ids.csv"
    
    RANDOM_STATE = 42
    TEST_SET_SIZE = 0.2

# --- 2. Parsing Function ---
def parse_lrec_line(line: str):
    parts = line.strip().split("\t")
    fname_indices = [i for i, p in enumerate(parts) if p.endswith(".txt")]
    if len(fname_indices) != 2: return None
    try:
        sentpair_id = int(parts[0])
        sent1 = " ".join(parts[fname_indices[0] + 2 : fname_indices[1]]).strip('"')
        sent2 = " ".join(parts[fname_indices[1] + 2 : len(parts) - 2]).strip('"')
        label = parts[-1]
        return {"sentpair_id": sentpair_id, "sent1": sent1, "sent2": sent2, "label": label}
    except (ValueError, IndexError): return None

# --- 3. Strict Data Loading (Train/Test Split Logic) ---
def load_full_data_aligned(config: Config):
    logging.info("Loading full dataset with Train/Test split alignment...")
    rows = []
    
    for fp in [config.POS_DATA_FILE, config.NEG_DATA_FILE]:
        if not os.path.exists(fp):
            logging.error(f"Data file not found at '{fp}'. Cannot proceed.")
            return None
        with open(fp, "r", encoding="utf-8") as f:
            for line in f:
                if parsed := parse_lrec_line(line):
                    rows.append(parsed)
                    
    df = pd.DataFrame(rows)
    if df.empty: return None
    
    LABEL_MAP = {"SUPPORT": 0, "ATTACK": 1, "NO_REL": 2, "NO_REL\n": 2}
    df["label_id"] = df["label"].str.strip().map(LABEL_MAP)
    df = df.dropna(subset=["sentpair_id", "sent1", "sent2", "label_id"])
    
    train_df, test_df = train_test_split(
        df, 
        test_size=config.TEST_SET_SIZE,
        random_state=config.RANDOM_STATE,
        stratify=df["label_id"]
    )
    
    df_combined_ordered = pd.concat([train_df, test_df]).reset_index(drop=True)
    df_combined_ordered['label_id'] = df_combined_ordered['label_id'].astype(int)
    df_combined_ordered['text'] = df_combined_ordered['sent1'] + " " + df_combined_ordered['sent2']
    
    logging.info(f"Dataset loaded. Total samples: {len(df_combined_ordered)}")
    return df_combined_ordered

# --- 4. Alignment Check Function ---
def check_alignment(df_text, id_csv_path, model_name="Model"):
    if not os.path.exists(id_csv_path):
        logging.error(f"ID file for {model_name} not found at {id_csv_path}")
        return False

    print(f"Checking alignment for {model_name}...")
    try:
        df_ids = pd.read_csv(id_csv_path)
        if 'sentpair_id' not in df_ids.columns:
             ids_from_file = df_ids.iloc[:, 0].astype(int).values
        else:
             ids_from_file = df_ids['sentpair_id'].astype(int).values    
    except Exception as e:
        logging.error(f"Failed to read ID file: {e}")
        return False

    ids_from_text = df_text['sentpair_id'].astype(int).values
    
    if len(ids_from_file) != len(ids_from_text):
        logging.warning(f"MISMATCH: Text rows ({len(ids_from_text)}) != ID rows ({len(ids_from_file)})")
        min_len = min(len(ids_from_file), len(ids_from_text))
        ids_from_file = ids_from_file[:min_len]
        ids_from_text = ids_from_text[:min_len]

    matches = (ids_from_file == ids_from_text).all()
    
    if matches:
        print(f"SUCCESS: {model_name} Data is perfectly aligned.")
        return True
    else:
        print(f"FAILURE: Alignment mismatch.")
        return False

# --- 5. BERTopic Execution with VISUALIZATION ---
def run_topic_modeling(df, emb_path, model_label):
    if not os.path.exists(emb_path):
        logging.error(f"Embedding file not found: {emb_path}")
        return

    print(f"\n{'='*20}\nRunning BERTopic for {model_label}\n{'='*20}")
    
    # Load Embeddings
    embeddings = np.load(emb_path)
    
    # Adjust lengths
    if len(embeddings) != len(df):
        min_len = min(len(embeddings), len(df))
        embeddings = embeddings[:min_len]
        docs = df['text'].iloc[:min_len].tolist()
    else:
        docs = df['text'].tolist()

    # Initialize UMAP & BERTopic
    # We use low n_neighbors to make distinct micro-clusters visible
    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
    
    topic_model = BERTopic(
        umap_model=umap_model,
        calculate_probabilities=False,
        verbose=True
    )
    
    # Train
    topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)
    
    # --- VISUALIZATIONS ---
    print("\nGenerating Visualizations... (Please wait for plots to render)")
    
    # 1. Intertopic Distance Map (The "Bubble Chart")
    #
    try:
        fig1 = topic_model.visualize_topics()
        fig1.show() # <--- THIS DISPLAYS IT IN THE NOTEBOOK
    except Exception as e:
        print(f"Could not display Distance Map: {e}")

    # 2. Topic Word Scores (Barchart)
    #
    try:
        fig2 = topic_model.visualize_barchart(top_n_topics=10)
        fig2.show() # <--- THIS DISPLAYS IT IN THE NOTEBOOK
    except Exception as e:
        print(f"Could not display Barchart: {e}")
        
    # 3. Hierarchy (Dendrogram)
    try:
        fig3 = topic_model.visualize_hierarchy()
        fig3.show() # <--- THIS DISPLAYS IT IN THE NOTEBOOK
    except Exception as e:
        print(f"Could not display Hierarchy: {e}")

# --- Main ---
if __name__ == "__main__":
    cfg = Config()
    df_aligned = load_full_data_aligned(cfg)
    
    if df_aligned is not None:
        # LBERT
        if check_alignment(df_aligned, cfg.LBERT_ID_PATH, "LBERT"):
            run_topic_modeling(df_aligned, cfg.LBERT_EMB_PATH, "LBERT")
        
        # InLBERT
        if check_alignment(df_aligned, cfg.INLBERT_ID_PATH, "InLBERT"):
            run_topic_modeling(df_aligned, cfg.INLBERT_EMB_PATH, "InLBERT")
    else:
        print("Critical: Data loading failed.")

2025-11-19 09:05:04.546358: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763543104.796287      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763543104.897386      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Checking alignment for LBERT...
SUCCESS: LBERT Data is perfectly aligned.

Running BERTopic for LBERT


2025-11-19 09:05:57,056 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-19 09:07:17,888 - BERTopic - Dimensionality - Completed ✓
2025-11-19 09:07:17,892 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-19 09:07:23,126 - BERTopic - Cluster - Completed ✓
2025-11-19 09:07:23,140 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-11-19 09:07:25,682 - BERTopic - Representation - Completed ✓



Generating Visualizations... (Please wait for plots to render)


Checking alignment for InLBERT...
SUCCESS: InLBERT Data is perfectly aligned.

Running BERTopic for InLBERT


2025-11-19 09:07:33,713 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-19 09:08:25,666 - BERTopic - Dimensionality - Completed ✓
2025-11-19 09:08:25,669 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-19 09:08:28,149 - BERTopic - Cluster - Completed ✓
2025-11-19 09:08:28,160 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-11-19 09:08:30,878 - BERTopic - Representation - Completed ✓



Generating Visualizations... (Please wait for plots to render)
